# CICIoT2023 공식 feature + MAC metadata 추출 v2

첫 테스트에서는 **DDoS-SYN_Flood.pcap 1개 + workers=1**만 사용합니다.

v2는 실패했을 때 `CalledProcessError`만 보여주는 대신,
**실제 내부 traceback과 stderr를 그대로 출력**합니다.


In [3]:
from google.colab import drive
drive.mount('/content/drive')


Mounted at /content/drive


In [12]:
# 필요한 패키지
!apt-get -qq update
!apt-get -qq install -y tcpdump
!pip -q install -U dpkt scapy scipy tqdm pandas


W: https://developer.download.nvidia.com/compute/cuda/repos/ubuntu2404/x86_64/InRelease: Key is stored in legacy trusted.gpg keyring (/etc/apt/trusted.gpg), see the DEPRECATION section in apt-key(8) for details.
W: Skipping acquire of configured file 'main/source/Sources' as repository 'https://r2u.stat.illinois.edu/ubuntu noble InRelease' does not seem to provide it (sources.list entry misspelt?)


In [13]:
# 환경 확인
from pathlib import Path
import shutil, sys
import dpkt, scapy, pandas as pd

BASE = Path('/content/drive/MyDrive/CICIoT2023')

PCAP = BASE / 'DDoS-SYN_Flood.pcap'

print('PCAP 존재:', PCAP.exists())
if PCAP.exists():
    print('PCAP 크기(GB):', round(PCAP.stat().st_size / 1024**3, 3))
print('tcpdump:', shutil.which('tcpdump'))
print('Python 실행파일:', sys.executable)
print('dpkt:', getattr(dpkt, '__version__', 'unknown'))
print('scapy:', getattr(scapy, '__version__', 'unknown'))
print('pandas:', pd.__version__)

if not PCAP.exists():
    raise FileNotFoundError(PCAP)


PCAP 존재: True
PCAP 크기(GB): 1.907
tcpdump: /usr/bin/tcpdump
Python 실행파일: /usr/bin/python3
dpkt: 1.9.8
scapy: 2.7.0
pandas: 3.0.6


In [14]:
from pathlib import Path
import sys
import shutil
import dpkt
import scapy
import pandas as pd

BASE = Path('/content/drive/MyDrive/CICIoT2023')
WORK = Path('/content/pcap2csv_metadata_v2')

PCAP = BASE / 'DDoS-SYN_Flood.pcap'

print("PCAP 존재:", PCAP.exists())

if PCAP.exists():
    print("PCAP 크기(GB):", round(PCAP.stat().st_size / (1024**3), 3))

print("실행 코드 존재:",
      (WORK / 'Generating_dataset_metadata.py').exists())

print("tcpdump:", shutil.which('tcpdump'))
print("Python:", sys.executable)
print("dpkt:", getattr(dpkt, '__version__', 'unknown'))
print("scapy:", getattr(scapy, '__version__', 'unknown'))
print("pandas:", pd.__version__)

PCAP 존재: True
PCAP 크기(GB): 1.907
실행 코드 존재: False
tcpdump: /usr/bin/tcpdump
Python: /usr/bin/python3
dpkt: 1.9.8
scapy: 2.7.0
pandas: 3.0.6


In [15]:
from pathlib import Path
import zipfile
import shutil

BASE = Path('/content/drive/MyDrive/CICIoT2023')
ZIP_PATH = BASE / 'pcap2csv_metadata_v2.zip'
WORK = Path('/content/pcap2csv_metadata_v2')

# 기존 폴더가 있으면 삭제
if WORK.exists():
    shutil.rmtree(WORK)

# zip 직접 압축 해제
with zipfile.ZipFile(ZIP_PATH, 'r') as z:
    z.extractall('/content')

print("압축파일:", ZIP_PATH)
print("WORK 존재:", WORK.exists())

# 실제 파일 확인
script = WORK / 'Generating_dataset_metadata.py'

print("실행 파일 존재:", script.exists())
print("실행 파일 경로:", script)

print("\n=== 압축 해제된 파일 ===")
for p in WORK.iterdir():
    print(p.name)

압축파일: /content/drive/MyDrive/CICIoT2023/pcap2csv_metadata_v2.zip
WORK 존재: True
실행 파일 존재: True
실행 파일 경로: /content/pcap2csv_metadata_v2/Generating_dataset_metadata.py

=== 압축 해제된 파일 ===
README_KR.txt
Communication_features.py
Generating_dataset.py
README_KR_v2.txt
Supporting_functions.py
Feature_extraction.py
Generating_dataset_metadata.py
Connectivity_features.py
Layered_features.py
Dynamic_features.py
Feature_extraction_metadata.py


In [16]:
from pathlib import Path

SCRIPT = Path(
    '/content/pcap2csv_metadata_v2/Generating_dataset_metadata.py'
)

text = SCRIPT.read_text(encoding='utf-8')

old = """['tcpdump', '-r', str(pcap), '-w', str(prefix), '-C', str(args.split_mb)]"""

new = """['tcpdump', '-Z', 'root', '-r', str(pcap), '-w', str(prefix), '-C', str(args.split_mb)]"""

if old in text:
    text = text.replace(old, new)
    SCRIPT.write_text(text, encoding='utf-8')
    print("✅ tcpdump 권한 수정 완료")
else:
    if "'-Z', 'root'" in text:
        print("✅ 이미 수정되어 있음")
    else:
        print("❌ 수정할 위치를 찾지 못함")

# 수정됐는지 확인
text = SCRIPT.read_text(encoding='utf-8')

for line in text.splitlines():
    if "'tcpdump'" in line:
        print(line)

✅ tcpdump 권한 수정 완료
    if shutil.which('tcpdump') is None:
            ['tcpdump', '-Z', 'root', '-r', str(pcap), '-w', str(prefix), '-C', str(args.split_mb)],


In [17]:
import subprocess
import sys
from pathlib import Path

BASE = Path('/content/drive/MyDrive/CICIoT2023')
WORK = Path('/content/pcap2csv_metadata_v2')

PCAP = BASE / 'DDoS-SYN_Flood.pcap'

OUT_DIR = BASE / 'features_with_metadata'
OUT_DIR.mkdir(parents=True, exist_ok=True)

OUT = OUT_DIR / 'DDoS-SYN_Flood_meta.csv'

# 이전 실패 결과가 혹시 있으면 제거
if OUT.exists():
    OUT.unlink()

cmd = [
    sys.executable,
    str(WORK / 'Generating_dataset_metadata.py'),
    '--pcap', str(PCAP),
    '--output', str(OUT),
    '--attack-type', 'DDoS-SYN_Flood',
    '--binary-label', '1',
    '--workers', '1'
]

print("실행 명령:")
print(" ".join(cmd))

result = subprocess.run(
    cmd,
    stdout=subprocess.PIPE,
    stderr=subprocess.STDOUT,
    text=True
)

print("\n===== pcap2csv 출력 =====")
print(result.stdout)

print("\nreturn code:", result.returncode)

if result.returncode == 0:
    print("\n✅ 추출 성공")
    print("저장:", OUT)
else:
    print("\n❌ 추출 실패")

실행 명령:
/usr/bin/python3 /content/pcap2csv_metadata_v2/Generating_dataset_metadata.py --pcap /content/drive/MyDrive/CICIoT2023/DDoS-SYN_Flood.pcap --output /content/drive/MyDrive/CICIoT2023/features_with_metadata/DDoS-SYN_Flood_meta.csv --attack-type DDoS-SYN_Flood --binary-label 1 --workers 1

===== pcap2csv 출력 =====
/usr/local/lib/python3.13/dist-packages/scapy/layers/tls/crypto/groups.py:25: CryptographyDeprecationWarning: Diffie-Hellman over finite fields (FFDH) is deprecated and support will be removed in a future release. Use a more modern key exchange algorithm.
  from cryptography.hazmat.primitives.asymmetric.dh import DHParameterNumbers
/content/pcap2csv_metadata_v2/Feature_extraction_metadata.py:551: RuntimeWarning: divide by zero encountered in divide
  sliced_df['Rate'] = num_of_packets/duration_time_interval
/content/pcap2csv_metadata_v2/Feature_extraction_metadata.py:551: RuntimeWarning: divide by zero encountered in divide
  sliced_df['Rate'] = num_of_packets/duration_tim

In [18]:
import pandas as pd

meta = pd.read_csv(OUT)

print("행 × 열:", meta.shape)

META_COLS = [
    'window_start_ts',
    'window_end_ts',
    'window_packet_count',
    'primary_src_mac',
    'primary_dst_mac',
    'src_macs',
    'dst_macs',
    'all_macs',
    'unique_mac_count',
    'source_pcap',
    'attack_type',
    'binary_label'
]

FEATURE_COLS = [
    c for c in meta.columns
    if c not in META_COLS
]

print("전체 컬럼 수:", len(meta.columns))
print("RF feature 수:", len(FEATURE_COLS))

display(
    meta[
        [
            'primary_src_mac',
            'primary_dst_mac',
            'all_macs',
            'attack_type',
            'binary_label'
        ]
    ].head(10)
)

행 × 열: (2659333, 51)
전체 컬럼 수: 51
RF feature 수: 39


,primary_src_mac,primary_dst_mac,all_macs,attack_type,binary_label
0,DC:A6:32:C9:E4:AB,1C:FE:2B:98:16:DD,1C:FE:2B:98:16:DD;DC:A6:32:C9:E4:AB,DDoS-SYN_Flood,1
1,E4:5F:01:55:90:C4,1C:FE:2B:98:16:DD,1C:FE:2B:98:16:DD;3C:18:A0:41:C3:A0;9C:8E:CD:1...,DDoS-SYN_Flood,1
2,E4:5F:01:55:90:C4,1C:FE:2B:98:16:DD,1C:FE:2B:98:16:DD;E4:5F:01:55:90:C4,DDoS-SYN_Flood,1
3,E4:5F:01:55:90:C4,1C:FE:2B:98:16:DD,1C:FE:2B:98:16:DD;DC:A6:32:C9:E4:D5;E4:5F:01:5...,DDoS-SYN_Flood,1
4,DC:A6:32:C9:E4:D5,1C:FE:2B:98:16:DD,1C:FE:2B:98:16:DD;DC:A6:32:C9:E4:D5,DDoS-SYN_Flood,1
5,DC:A6:32:C9:E4:D5,1C:FE:2B:98:16:DD,1C:FE:2B:98:16:DD;DC:A6:32:C9:E4:D5,DDoS-SYN_Flood,1
6,DC:A6:32:C9:E4:D5,1C:FE:2B:98:16:DD,1C:FE:2B:98:16:DD;DC:A6:32:C9:E4:D5,DDoS-SYN_Flood,1
7,DC:A6:32:C9:E4:D5,1C:FE:2B:98:16:DD,1C:FE:2B:98:16:DD;DC:A6:32:C9:E4:D5,DDoS-SYN_Flood,1
8,DC:A6:32:C9:E4:D5,1C:FE:2B:98:16:DD,1C:FE:2B:98:16:DD;DC:A6:32:C9:E4:D5,DDoS-SYN_Flood,1
9,DC:A6:32:C9:E4:D5,1C:FE:2B:98:16:DD,1C:FE:2B:98:16:DD;DC:A6:32:C9:E4:D5,DDoS-SYN_Flood,1


In [19]:
# Camera / Echo Dot 1 / AMCREST window 수
CAMERA_MACS = {
    '9C:8E:CD:1D:AB:9F','3C:37:86:6F:B9:51','40:5D:82:35:14:C8',
    'C0:E7:BF:0A:79:D1','B0:C5:54:59:2E:99','44:01:BB:EC:10:4A',
    '34:75:63:73:F3:36','7C:A7:B0:CD:18:32','44:BB:3B:00:39:07',
    '70:EE:50:68:0E:32','10:5A:17:97:A5:C6','10:2C:6B:1B:43:BE',
    '6C:5A:B0:44:1D:90','7C:78:B2:86:0D:81','84:7A:B6:64:62:58',
    '84:7A:B6:62:3A:6C','2C:D2:6B:66:D2:87'
}
ECHO_DOT1 = '1C:FE:2B:98:16:DD'
AMCREST = '9C:8E:CD:1D:AB:9F'

def mac_set(x):
    if pd.isna(x) or not str(x).strip():
        return set()
    return set(str(x).split(';'))

window_macs = meta['all_macs'].apply(mac_set)

print('전체 windows:', len(meta))
print('Camera windows:', int(window_macs.apply(lambda s: bool(s & CAMERA_MACS)).sum()))
print('Echo Dot 1 windows:', int(window_macs.apply(lambda s: ECHO_DOT1 in s).sum()))
print('AMCREST windows:', int(window_macs.apply(lambda s: AMCREST in s).sum()))


전체 windows: 2659333
Camera windows: 325179
Echo Dot 1 windows: 797226
AMCREST windows: 294077


In [4]:
from pathlib import Path
import pandas as pd
import numpy as np

OUT = Path(
    '/content/drive/MyDrive/CICIoT2023/features_with_metadata/'
    'DDoS-SYN_Flood_meta.csv'
)

print("파일 존재:", OUT.exists())

meta = pd.read_csv(OUT)

META_COLS = [
    'window_start_ts',
    'window_end_ts',
    'window_packet_count',
    'primary_src_mac',
    'primary_dst_mac',
    'src_macs',
    'dst_macs',
    'all_macs',
    'unique_mac_count',
    'source_pcap',
    'attack_type',
    'binary_label'
]

FEATURE_COLS = [
    c for c in meta.columns
    if c not in META_COLS
]

print("불러오기 완료")
print("shape:", meta.shape)
print("RF feature 수:", len(FEATURE_COLS))

파일 존재: True
불러오기 완료
shape: (2659333, 51)
RF feature 수: 39


In [5]:
import numpy as np

print("window_packet_count 분포:")
print(meta['window_packet_count'].value_counts().sort_index())

print("\nall_macs 비어있는 행:",
      meta['all_macs'].isna().sum())

X = meta[FEATURE_COLS]
numeric_X = X.select_dtypes(include=np.number)

print("\ninf 개수:",
      np.isinf(numeric_X).sum().sum())

print("NaN 개수:",
      X.isna().sum().sum())

print("\nlabel:")
print(meta['binary_label'].value_counts())

print("\nattack type:")
print(meta['attack_type'].value_counts())

window_packet_count 분포:
window_packet_count
1          22
2          19
3          17
4          20
5          20
6          17
7          22
8          24
9          20
10    2659152
Name: count, dtype: int64

all_macs 비어있는 행: 0

inf 개수: 80
NaN 개수: 44

label:
binary_label
1    2659333
Name: count, dtype: int64

attack type:
attack_type
DDoS-SYN_Flood    2659333
Name: count, dtype: int64
